# Before and after: missing-data fixes of 2026-10-10

Compares the cleaned data and scenarios from before the fixes (archived in
`archive/data_2026-10-09_before_missing_data_fixes/`) with the current outputs of
`data_prep.ipynb`. Run `data_prep.ipynb` first so the "after" files are current.

The fixes (`docs/DATA_PREP_REPORT.md`, section 10):

1. **Bug:** long gaps in a VM's usage were filled at 100% of every requested vCPU instead
   of being left out. DuckDB's `least()` ignores NULLs.
2. **Collector outages:** slots when the collector was down for every VM now count
   toward $W_i$ at the VM's own mean usage. Before, they were skipped.
3. **Unmonitored hosts:** VMs on the 26 hosts that never report usage are removed by a
   named step. Imputing them is optional and off, so the workload set is unchanged.

Model formulas did not change. Only input values did.

In [1]:
import json
from pathlib import Path

import duckdb

# Repository root: the nearest folder up from here that holds pyproject.toml.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
BEFORE = ROOT / "archive" / "data_2026-10-09_before_missing_data_fixes"
AFTER_CLEAN, AFTER_SCEN = ROOT / "data" / "clean", ROOT / "scenarios"
assert BEFORE.exists(), "run this notebook from the repository root"

con = duckdb.connect()
con.execute(f'''
    CREATE VIEW wb AS SELECT * FROM '{BEFORE / "clean" / "workloads.csv"}';
    CREATE VIEW wa AS SELECT * FROM '{AFTER_CLEAN / "workloads.csv"}';
    CREATE VIEW ub AS SELECT * FROM '{BEFORE / "clean" / "workload_usage.parquet"}';
    CREATE VIEW ua AS SELECT * FROM '{AFTER_CLEAN / "workload_usage.parquet"}';
''');

## 1. Cleaning log

The VM counts are the same at every step. The "after" log names the
unmonitored-host step, which the "before" log folded into "has any usage reading".

In [2]:
con.sql(f'''
    WITH b AS (SELECT row_number() OVER () AS n, step, unit, after FROM '{BEFORE / "clean" / "cleaning_log_workloads.csv"}'),
         a AS (SELECT row_number() OVER () AS n, step, unit, after FROM '{AFTER_CLEAN / "cleaning_log_workloads.csv"}')
    SELECT coalesce(a.step, b.step) AS step, coalesce(a.unit, b.unit) AS unit,
           b.after AS remaining_before, a.after AS remaining_after
    FROM a FULL JOIN b USING (step) ORDER BY coalesce(a.n, b.n + 0.5)''')

┌───────────────────────────────────────┬─────────┬──────────────────┬─────────────────┐
│                 step                  │  unit   │ remaining_before │ remaining_after │
│                varchar                │ varchar │      int64       │      int64      │
├───────────────────────────────────────┼─────────┼──────────────────┼─────────────────┤
│ snap to 5-minute slots, merge doubles │ rows    │         15395669 │        15395669 │
│ usage reported as NA                  │ rows    │         11942874 │        11942874 │
│ has a flavor record                   │ VMs     │             2799 │            2799 │
│ drop infrastructure flavors           │ VMs     │             1193 │            1193 │
│ drop implausible flavor memory        │ VMs     │             1191 │            1191 │
│ drop resized VMs                      │ VMs     │             1172 │            1172 │
│ drop migrated VMs                     │ VMs     │             1166 │            1166 │
│ drop VMs alive at t

## 2. How missing usage slots were filled

"Before" marked a slot only as filled or not. A filled slot that sits exactly at the
flavor ceiling (100% × vCPUs) is the bug: a long gap given full load. "After" records
the kind of every slot. Long gaps while the collector was up are now left out of the
work window, so they no longer appear here.

In [3]:
con.sql('''
    WITH b AS (
        SELECT CASE WHEN NOT u.filled THEN 'observed'
                    WHEN u.cpu_pct = 100.0 * w.C_req_vcpu THEN 'long gap set to full load (bug)'
                    ELSE 'interpolated' END AS fill,
               count(*) AS slots_before, count(DISTINCT u.workload_id) AS workloads_before
        FROM ub u JOIN wb w USING (workload_id) GROUP BY 1),
    a AS (SELECT fill, count(*) AS slots_after, count(DISTINCT workload_id) AS workloads_after FROM ua GROUP BY 1)
    SELECT fill, slots_before, slots_after, workloads_before, workloads_after
    FROM b FULL JOIN a USING (fill) ORDER BY coalesce(slots_before, 0) + coalesce(slots_after, 0) DESC''')

┌─────────────────────────────────┬──────────────┬─────────────┬──────────────────┬─────────────────┐
│              fill               │ slots_before │ slots_after │ workloads_before │ workloads_after │
│             varchar             │    int64     │    int64    │      int64       │      int64      │
├─────────────────────────────────┼──────────────┼─────────────┼──────────────────┼─────────────────┤
│ observed                        │      2758428 │     2758428 │              502 │             502 │
│ outage_mean                     │         NULL │       29543 │             NULL │             108 │
│ long gap set to full load (bug) │        12633 │        NULL │              165 │            NULL │
│ interpolated                    │          477 │         989 │              144 │             174 │
└─────────────────────────────────┴──────────────┴─────────────┴──────────────────┴─────────────────┘

## 3. Work content per workload ($W_i$)

Same 502 workloads before and after, matched on the UFCG VM id. Each is put in the group
of the fix that touched it:

- **bug removed:** had long-gap slots at full load before
- **outage filled:** has collector-outage slots filled at its mean usage after
- **both** or **neither**

A few "neither" workloads still rise slightly (well under 1%). They span collector gaps
of up to 3 slots, which are now interpolated like any other short gap. Before, those
slots were skipped.

In [4]:
con.execute('''
    CREATE TEMP TABLE cmp AS
    WITH bug AS (
        SELECT u.workload_id, count(*) AS bug_slots
        FROM ub u JOIN wb w USING (workload_id)
        WHERE u.filled AND u.cpu_pct = 100.0 * w.C_req_vcpu GROUP BY 1)
    SELECT a.workload_id, a.flavor_name, a.C_req_vcpu, a.lifetime_slots,
           b.W_MHz_s AS W_before, a.W_MHz_s AS W_after, a.W_MHz_s / b.W_MHz_s - 1 AS rel_change,
           b.ET_base_periods AS ET_before, a.ET_base_periods AS ET_after,
           b.mean_util AS util_before, a.mean_util AS util_after,
           coalesce(bug.bug_slots, 0) AS bug_slots, a.n_outage_slots AS outage_slots,
           CASE WHEN bug.bug_slots > 0 AND a.n_outage_slots > 0 THEN 'both'
                WHEN bug.bug_slots > 0 THEN 'bug removed'
                WHEN a.n_outage_slots > 0 THEN 'outage filled'
                ELSE 'neither' END AS cause
    FROM wa a JOIN wb b USING (ufcg_server_id)
    LEFT JOIN bug ON bug.workload_id = b.workload_id''')
print("workloads matched:", con.sql("SELECT count(*) FROM cmp").fetchone()[0])
con.sql('''
    SELECT cause, count(*) AS workloads,
           count(*) FILTER (rel_change < -1e-6) AS W_down, count(*) FILTER (rel_change > 1e-6) AS W_up,
           round(100 * median(rel_change), 1) AS median_change_pct,
           round(100 * min(rel_change), 1) AS min_change_pct, round(100 * max(rel_change), 1) AS max_change_pct,
           count(*) FILTER (ET_after <> ET_before) AS ET_changed
    FROM cmp GROUP BY 1 ORDER BY workloads DESC''')

workloads matched: 502


┌───────────────┬───────────┬────────┬───────┬───────────────────┬────────────────┬────────────────┬────────────┐
│     cause     │ workloads │ W_down │ W_up  │ median_change_pct │ min_change_pct │ max_change_pct │ ET_changed │
│    varchar    │   int64   │ int64  │ int64 │      double       │     double     │     double     │   int64    │
├───────────────┼───────────┼────────┼───────┼───────────────────┼────────────────┼────────────────┼────────────┤
│ neither       │       287 │      0 │    22 │               0.0 │            0.0 │            0.4 │          3 │
│ bug removed   │       107 │    107 │     0 │             -16.0 │          -97.8 │           -0.0 │        107 │
│ both          │        58 │     35 │    23 │              -0.6 │          -55.6 │           82.2 │         51 │
│ outage filled │        50 │      0 │    50 │               4.2 │            0.1 │          118.4 │         42 │
└───────────────┴───────────┴────────┴───────┴───────────────────┴────────────────┴─────

In [5]:
# How big the changes are
con.sql('''
    SELECT CASE WHEN rel_change <= -0.5 THEN '1: fell by more than half'
                WHEN rel_change <= -0.1 THEN '2: fell 10-50%'
                WHEN rel_change < -1e-6 THEN '3: fell under 10%'
                WHEN rel_change <= 1e-6 THEN '4: unchanged'
                WHEN rel_change < 0.1 THEN '5: rose under 10%'
                WHEN rel_change < 0.5 THEN '6: rose 10-50%'
                ELSE '7: rose by more than half' END AS change_in_W,
           count(*) AS workloads
    FROM cmp GROUP BY 1 ORDER BY 1''')

┌───────────────────────────┬───────────┐
│        change_in_W        │ workloads │
│          varchar          │   int64   │
├───────────────────────────┼───────────┤
│ 1: fell by more than half │        26 │
│ 2: fell 10-50%            │        46 │
│ 3: fell under 10%         │        70 │
│ 4: unchanged              │       265 │
│ 5: rose under 10%         │        90 │
│ 6: rose 10-50%            │         3 │
│ 7: rose by more than half │         2 │
└───────────────────────────┴───────────┘

In [6]:
# Totals across the catalogue
con.sql('''
    SELECT round(sum(W_before) / 1e9, 2) AS total_W_before_THz_s, round(sum(W_after) / 1e9, 2) AS total_W_after_THz_s,
           round(100 * (sum(W_after) / sum(W_before) - 1), 1) AS change_pct,
           round(100 * median(util_before), 2) AS median_util_before_pct,
           round(100 * median(util_after), 2) AS median_util_after_pct
    FROM cmp''')

┌──────────────────────┬─────────────────────┬────────────┬────────────────────────┬───────────────────────┐
│ total_W_before_THz_s │ total_W_after_THz_s │ change_pct │ median_util_before_pct │ median_util_after_pct │
│        double        │       double        │   double   │         double         │        double         │
├──────────────────────┼─────────────────────┼────────────┼────────────────────────┼───────────────────────┤
│               500.77 │              480.01 │       -4.1 │                   3.55 │                  2.67 │
└──────────────────────┴─────────────────────┴────────────┴────────────────────────┴───────────────────────┘

In [7]:
# The 15 largest changes, either way
con.sql('''
    SELECT workload_id, flavor_name, lifetime_slots, cause, bug_slots, outage_slots,
           round(W_before / 1e6, 1) AS W_before_GHz_s, round(W_after / 1e6, 1) AS W_after_GHz_s,
           round(100 * rel_change, 1) AS change_pct, ET_before, ET_after
    FROM cmp ORDER BY abs(ln(W_after / W_before)) DESC LIMIT 15''')

┌─────────────┬────────────────┬────────────────┬─────────────┬───────────┬──────────────┬────────────────┬───────────────┬────────────┬───────────┬──────────┐
│ workload_id │  flavor_name   │ lifetime_slots │    cause    │ bug_slots │ outage_slots │ W_before_GHz_s │ W_after_GHz_s │ change_pct │ ET_before │ ET_after │
│   varchar   │    varchar     │     int64      │   varchar   │   int64   │    int64     │     double     │    double     │   double   │   int64   │  int64   │
├─────────────┼────────────────┼────────────────┼─────────────┼───────────┼──────────────┼────────────────┼───────────────┼────────────┼───────────┼──────────┤
│ w0051       │ general.medium │             11 │ bug removed │         1 │            0 │            1.5 │           0.0 │      -97.8 │         2 │        1 │
│ w0049       │ general.medium │             11 │ bug removed │         1 │            0 │            1.5 │           0.0 │      -97.7 │         2 │        1 │
│ w0473       │ general.medium │        

## 4. Baseline execution time and the scenario catalogue

Scenarios admit only workloads with a baseline execution time of 3 to 144 periods
(`small`: 3 to 36). A changed $ET^{base}_i$ can move a workload across that line.

In [8]:
con.sql('''
    WITH k AS (
        SELECT 'before' AS version, ET_before AS et FROM cmp UNION ALL SELECT 'after', ET_after FROM cmp)
    PIVOT (SELECT version, CASE WHEN et < 3 THEN '1: under 3 periods'
                                WHEN et <= 12 THEN '2: 3-12 periods'
                                WHEN et <= 36 THEN '3: 13-36 periods'
                                WHEN et <= 144 THEN '4: 37-144 periods'
                                WHEN et <= 288 THEN '5: 145-288 periods'
                                ELSE '6: more than 1 day' END AS baseline_execution_time FROM k)
    ON version IN ('before', 'after') USING count(*) ORDER BY baseline_execution_time''')

┌─────────────────────────┬────────┬───────┐
│ baseline_execution_time │ before │ after │
│         varchar         │ int64  │ int64 │
├─────────────────────────┼────────┼───────┤
│ 1: under 3 periods      │    123 │   131 │
│ 2: 3-12 periods         │    123 │   115 │
│ 3: 13-36 periods        │     48 │    50 │
│ 4: 37-144 periods       │     73 │    75 │
│ 5: 145-288 periods      │     31 │    32 │
│ 6: more than 1 day      │    104 │    99 │
└─────────────────────────┴────────┴───────┘

In [9]:
con.sql('''
    SELECT count(*) FILTER (ET_before BETWEEN 3 AND 144) AS admitted_before,
           count(*) FILTER (ET_after BETWEEN 3 AND 144) AS admitted_after,
           count(*) FILTER (ET_after BETWEEN 3 AND 144 AND NOT ET_before BETWEEN 3 AND 144) AS newly_admitted,
           count(*) FILTER (ET_before BETWEEN 3 AND 144 AND NOT ET_after BETWEEN 3 AND 144) AS newly_excluded
    FROM cmp''')

┌─────────────────┬────────────────┬────────────────┬────────────────┐
│ admitted_before │ admitted_after │ newly_admitted │ newly_excluded │
│      int64      │     int64      │     int64      │     int64      │
├─────────────────┼────────────────┼────────────────┼────────────────┤
│             244 │            240 │              6 │             10 │
└─────────────────┴────────────────┴────────────────┴────────────────┘

## 5. Scenarios

Scenario diagnostics from each `params.json`. Servers are identical before and after
(section 6). Every difference comes from the workloads.

In [10]:
names = ["base", "thesis10", "small"]
diag = {}
for n in names:
    for ver, root in (("before", BEFORE / "scenarios"), ("after", AFTER_SCEN)):
        with open(root / n / "params.json") as f:
            diag[n, ver] = json.load(f)["diagnostics"]

keys = list(dict.fromkeys(k for d in diag.values() for k in d))   # a key missing from an older file shows as "-"
print(f"{'':<44}" + "".join(f"{n + ' before':>18}{n + ' after':>18}" for n in names))
for k in keys:
    row = "".join(f"{str(diag[n, 'before'].get(k, '-')):>18}{str(diag[n, 'after'].get(k, '-')):>18}" for n in names)
    flag = "" if all(diag[n, "before"].get(k) == diag[n, "after"].get(k) for n in names) else "  *"
    print(f"{k:<44}{row}{flag}")
print("\n* = differs in at least one scenario")

                                                   base before        base after   thesis10 before    thesis10 after      small before       small after
I_workloads                                                242               238               242               238               102                99  *
S_servers                                                    6                 6                10                10                 3                 3
Z_zones                                                      2                 2                 2                 2                 2                 2
T_periods                                                  288               288               288               288                96                96
Y_ist_binaries_inside_windows                            94176             91812            156960            153020              6846              6762  *
catalogue_size_within_ET_limits                            244              

In [11]:
# Which workloads each scenario draws. The draw is seeded, but a workload that changed
# its execution time can enter or leave the catalogue and shift everything after it.
# RT_i also differs for a second reason: the "before" files drew it from the same random
# sequence as the draw order, while it is now a seeded hash of each workload's VM id
# (stable from now on). A changed RT_i moves the deadline with it.
rows = []
for n in names:
    b = f"'{BEFORE / 'scenarios' / n / 'workloads.csv'}'"
    a = f"'{AFTER_SCEN / n / 'workloads.csv'}'"
    rows.append(con.sql(f'''
        SELECT '{n}' AS scenario,
               (SELECT count(*) FROM {b}) AS workloads_before, (SELECT count(*) FROM {a}) AS workloads_after,
               count(*) FILTER (b.workload_id IS NOT NULL AND a.workload_id IS NOT NULL) AS in_both,
               count(*) FILTER (a.workload_id IS NULL) AS only_before,
               count(*) FILTER (b.workload_id IS NULL) AS only_after,
               count(*) FILTER (a.ET_base_i <> b.ET_base_i) AS in_both_ET_changed,
               count(*) FILTER (a.RT_i <> b.RT_i) AS in_both_RT_changed,
               count(*) FILTER (a.D_i <> b.D_i) AS in_both_deadline_changed
        FROM {b} b FULL JOIN {a} a USING (workload_id)''').fetchall()[0])
print(f"{'scenario':<10}{'before':>8}{'after':>8}{'in both':>9}{'only before':>13}{'only after':>12}"
      f"{'ET changed':>12}{'RT changed':>12}{'deadline changed':>18}")
for r in rows:
    print(f"{r[0]:<10}{r[1]:>8}{r[2]:>8}{r[3]:>9}{r[4]:>13}{r[5]:>12}{r[6]:>12}{r[7]:>12}{r[8]:>18}")

scenario    before   after  in both  only before  only after  ET changed  RT changed  deadline changed
base           242     238      232           10           6          72         102               127
thesis10       242     238      232           10           6          72         102               127
small          102      99       99            3           0          23          43                52


## 6. Servers

The fixes touched only the workload side. This confirms the server files are
byte-for-byte identical.

In [12]:
pairs = [(BEFORE / "clean" / "servers.csv", AFTER_CLEAN / "servers.csv")] + \
        [(BEFORE / "scenarios" / n / "servers.csv", AFTER_SCEN / n / "servers.csv") for n in names]
for b, a in pairs:
    print(f"{'identical' if b.read_bytes() == a.read_bytes() else 'DIFFERENT':<10} {a.relative_to(ROOT)}")

identical  data/clean/servers.csv
identical  scenarios/base/servers.csv
identical  scenarios/thesis10/servers.csv
identical  scenarios/small/servers.csv
